# 05. 문서 레이아웃과 DocVQA 미니 베이스라인

OCR 텍스트만 이어 붙이면 테이블/영수증/청구서에서 위치 정보를 잃습니다.
이 노트북은 bounding box를 가진 OCR 토큰을 만들고, 텍스트 검색 베이스라인과 레이아웃 힌트 베이스라인을 비교합니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
tokens = pd.DataFrame([
    ("invoice", 30, 20, 100, 40, "header"),
    ("no", 110, 20, 135, 40, "header"),
    ("a17", 145, 20, 180, 40, "header"),
    ("item", 30, 80, 80, 100, "table_header"),
    ("price", 240, 80, 300, 100, "table_header"),
    ("sensor", 30, 120, 120, 140, "row"),
    ("42000", 240, 120, 300, 140, "row"),
    ("status", 30, 170, 95, 190, "footer"),
    ("paid", 240, 170, 290, 190, "footer"),
], columns=["text", "x1", "y1", "x2", "y2", "region"])

questions = pd.DataFrame([
    ("문서 번호는?", "a17", "header"),
    ("가격은?", "42000", "row"),
    ("상태는?", "paid", "footer"),
], columns=["question", "answer", "target_region"])

tokens


In [ ]:
def text_only_answer(question, tokens):
    q = question.lower()
    if "번호" in q or "no" in q:
        candidates = tokens[tokens["text"].str.contains("a|[0-9]", regex=True)]
    elif "가격" in q or "price" in q:
        candidates = tokens[tokens["text"].str.contains("[0-9]", regex=True)]
    elif "상태" in q or "status" in q:
        candidates = tokens[tokens["text"].isin(["paid", "unpaid", "done"])]
    else:
        candidates = tokens
    return candidates.iloc[-1]["text"] if len(candidates) else ""

def layout_answer(question, tokens, target_region):
    region_tokens = tokens[tokens["region"] == target_region]
    return text_only_answer(question, region_tokens if len(region_tokens) else tokens)

rows = []
for _, row in questions.iterrows():
    pred_text = text_only_answer(row.question, tokens)
    pred_layout = layout_answer(row.question, tokens, row.target_region)
    rows.append({
        "question": row.question,
        "answer": row.answer,
        "text_only": pred_text,
        "layout_aware": pred_layout,
        "text_only_em": pred_text == row.answer,
        "layout_em": pred_layout == row.answer,
    })
docqa_eval = pd.DataFrame(rows)
docqa_eval.to_csv(ARTIFACTS / "docqa_mini_eval.csv", index=False, encoding="utf-8-sig")
docqa_eval


In [ ]:
if plt:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.set_xlim(0, 340)
    ax.set_ylim(220, 0)
    for _, r in tokens.iterrows():
        rect = plt.Rectangle((r.x1, r.y1), r.x2-r.x1, r.y2-r.y1, fill=False)
        ax.add_patch(rect)
        ax.text(r.x1, r.y1 - 2, r.text, fontsize=9)
    ax.set_title("OCR tokens with layout boxes")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(ARTIFACTS / "docqa_layout_boxes.png", dpi=160)


## 논문 연결

최종 논문에서 `텍스트만 사용한 QA`와 `레이아웃/품질 힌트를 넣은 QA`를 비교합니다.
실제 데이터로 확장할 때는 DocVQA, TextVQA, OCRBench 계열의 태스크 정의를 참고하세요.
